In [ ]:
import os
import sys

# to setup import paths add project root dirs to sys.path
sys.path.append(os.path.join(os.getcwd(), "..", ".."))
from baseVR.base_functionality import init_import_paths
init_import_paths()

In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

%matplotlib widget
from matplotlib.colors import Normalize

from analytics_processing import analytics
import analytics_processing.analytics_constants as C
from CustomLogger import CustomLogger as Logger

from analytics_processing.sessions_from_nas_parsing import sessionlist_fullfnames_from_args
from analytics_processing.sessions_from_nas_parsing import fullfnames2snames

from analytics_processing.modality_loading import session_modality_from_nas


import scipy.stats as stats

In [ ]:
# first set the paths and logger. You can see all kind of outputs in this directory 
# already. Poster used plots from this directory.

data = {}
nas_dir = C.device_paths()[0]
output_dir = f"../outputs/validation/lick_check/"
Logger().init_logger(None, None, logging_level="WARNING")

In [ ]:
# analysis was intially done on all sessions from animal 6 with paradigm 1100
# But the poster focused on S8-15

# animal_ids = [10]
# animal_ids = [5]
animal_ids = [7]
paradigm_ids = [800]
session_ids = None

In [ ]:
session_dirs, _ = sessionlist_fullfnames_from_args(paradigm_ids, animal_ids, session_ids)
session_names = fullfnames2snames(session_dirs)

In [ ]:
beh = analytics.get_analytics('BehaviorTrackwise', session_names=session_names)#[:3])
beh.index = beh.index.droplevel((0,3))
beh.set_index('trial_id', append=True, inplace=True)

beh.columns

In [ ]:
beh[['from_position_bin', 'lick_detected', 'cue', 'reward-valve-open_count']]#.unstack('from_position_bin')

In [ ]:

def plot_lick_scatter_by_cue(
    df: pd.DataFrame,
    x_col: str = "from_position_bin",
    lick_col: str = "lick_detected",
    cue_col: str = "cue",
    reward_col: str = "reward-valve-open_count",
    session_level: str = "session_id",
    trial_level: str = "trial_id",
    cue_values=(1, 2),
    only_licks: bool = True,
    lick_threshold: float = 0.0,
    reward_threshold: float = 0.0,
    point_size: float = 4,
    reward_point_size: float = 24,
    alpha: float = 0.65,
    figsize=(15, 20),
    zone_ranges=((70,115 ), (195, 240)),
    zone_color: str = "tab:orange",
    zone_alpha: float = 0.2,
):
    required_cols = {x_col, lick_col, cue_col, reward_col}
    missing_cols = required_cols - set(df.columns)
    if missing_cols:
        raise ValueError(f"Missing required columns: {sorted(missing_cols)}")

    work = df.copy()
    if isinstance(work.index, pd.MultiIndex):
        missing_levels = {session_level, trial_level} - set(work.index.names)
        if missing_levels:
            raise ValueError(
                f"Missing required index levels: {sorted(missing_levels)}. "
                f"Current index levels: {work.index.names}"
            )
        work = work.reset_index()
    else:
        missing_id_cols = {session_level, trial_level} - set(work.columns)
        if missing_id_cols:
            raise ValueError(
                f"{session_level} and {trial_level} must be index levels or columns. "
                f"Missing: {sorted(missing_id_cols)}"
            )

    work = work.dropna(subset=[x_col, lick_col, cue_col, reward_col, session_level, trial_level]).copy()

    trial_order = (
        work[[session_level, trial_level]]
        .drop_duplicates()
        .sort_values([session_level, trial_level], kind="mergesort")
        .reset_index(drop=True)
    )
    trial_order["y"] = np.arange(len(trial_order), dtype=float)

    work = work.merge(
        trial_order,
        on=[session_level, trial_level],
        how="left",
        validate="m:1",
    )

    session_counts = trial_order.groupby(session_level, sort=False).size()
    session_starts = np.r_[0, session_counts.cumsum().to_numpy()[:-1]]
    session_ends = session_counts.cumsum().to_numpy() - 1
    session_centers = (session_starts + session_ends) / 2
    session_boundaries = session_counts.cumsum().to_numpy()[:-1] - 0.5

    fig, axes = plt.subplots(
        1,
        2,
        figsize=figsize,
        sharey=True,
        constrained_layout=True,
        gridspec_kw={"wspace": 0.04},
    )

    for ax, cue_val in zip(axes, cue_values):
        panel = work[work[cue_col] == cue_val]

        for x0, x1 in zone_ranges:
            ax.axvspan(x0, x1, color=zone_color, alpha=zone_alpha, zorder=0)

        if only_licks:
            panel_to_plot = panel[panel[lick_col] > lick_threshold]
            ax.scatter(
                panel_to_plot[x_col],
                panel_to_plot["y"],
                s=point_size,
                alpha=alpha,
                color="black",
                linewidths=0,
                zorder=2,
            )
        else:
            panel_to_plot = panel
            ax.scatter(
                panel_to_plot[x_col],
                panel_to_plot["y"],
                s=point_size,
                alpha=alpha,
                c=panel_to_plot[lick_col],
                cmap="viridis",
                linewidths=0,
                zorder=2,
            )

        reward_panel = panel_to_plot[panel_to_plot[reward_col] > reward_threshold]
        ax.scatter(
            reward_panel[x_col],
            reward_panel["y"],
            s=reward_point_size,
            facecolors="none",
            edgecolors="limegreen",
            linewidths=1.0,
            marker="o",
            zorder=3,
        )

        for yb in session_boundaries:
            ax.axhline(yb, color="gray", lw=0.8, alpha=0.8, zorder=1)

        n_trials = panel[[session_level, trial_level]].drop_duplicates().shape[0]
        ax.set_title(f"{cue_col} = {cue_val} (trials: {n_trials})")
        ax.set_xlabel(x_col)
        ax.set_ylim(-0.5, len(trial_order) - 0.5)

    axes[0].set_ylabel(f"{session_level} / {trial_level} (ordered by session, then trial)")
    axes[0].set_yticks(session_centers)
    axes[0].set_yticklabels(session_counts.index.astype(str).tolist())

    return fig, axes, trial_order


fig, axes, trial_order = plot_lick_scatter_by_cue(
    beh[["from_position_bin", "lick_detected", "cue", "reward-valve-open_count"]]
)
plt.show()

In [ ]:
fig, axes, trial_order = plot_lick_scatter_by_cue(
    beh[["from_position_bin", "lick_detected", "cue", "reward-valve-open_count"]]
)
plt.show()